# Phase 3: PlantVillage and PlantDoc preliminary image audit

This notebook accompanies `scripts/analyze_data.py`. It reports **measured** image counts, class imbalance, dimensions, unreadable files, exact duplicates, related-leaf indicators, and case-insensitive path collisions. The original dataset category names are retained; category matching is a separate task. No model is trained, and the raw images are never modified.

The charts in the assignment are examples, **not results**. Execute these cells against the prepared local datasets before drawing conclusions. File readability and identical byte content do not validate disease labels.


## 1. Run the inventory

Run this once after `python scripts/prepare_data.py`; rerun if the underlying files change. Set `DATA_ROOT` if using another machine. The script records source versions from the local manifest and scans *all supported image files* to count categories, decode image dimensions and modes, flag unreadable files, and identify exact duplicates and related-leaf groups.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "analyze_data.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # notebook opened from scripts/
DATA_ROOT = Path.home() / "datasets" / "cecs551-plant-disease"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "eda"
subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "analyze_data.py"),
                "--data-root", str(DATA_ROOT), "--output-dir", str(OUTPUT_DIR)], check=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

inventory = pd.read_csv(OUTPUT_DIR / "image_inventory.csv", keep_default_na=False)
counts = pd.read_csv(OUTPUT_DIR / "class_counts.csv", keep_default_na=False)
flags = pd.read_csv(OUTPUT_DIR / "quality_flags.csv", keep_default_na=False)
summary = json.loads((OUTPUT_DIR / "summary.json").read_text())
print("PlantVillage revision:", summary["plantvillage_revision"])
print("PlantDoc revision:", summary["plantdoc_revision"])
print("Images counted:", summary["images_by_dataset"])
print("Case-insensitive filename collision groups:", summary["case_insensitive_filename_collision_groups"])
DATASET_LABELS = {"plantvillage": "PlantVillage", "plantdoc": "PlantDoc"}
display(counts)


## 2. Class balance by original category

Each bar is the measured total for one category, with its original train/test membership shown separately. These are **file counts**, including any unreadable files; the table above also records readable counts.


In [ ]:
for dataset, display_name in DATASET_LABELS.items():
    subset = counts[counts["dataset"] == dataset]
    pivot = subset.pivot_table(index="category", columns="split", values="file_count",
                               aggfunc="sum", fill_value=0)
    ax = pivot.sort_values(by=list(pivot.columns), ascending=False).plot(
        kind="bar", stacked=True, figsize=(max(11, len(pivot) * .37), 5.5))
    ax.set(title=f"{display_name}: images per source category / split", ylabel="Image files", xlabel="Original category")
    ax.legend(title="Source split")
    plt.xticks(rotation=75, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{dataset.lower()}_class_balance.png", dpi=160)
    plt.show()


## 3. Image dimensions, unreadable files, duplicates, and leaf groups

The dataset-wide tables below show minimum, median, and maximum image width/height, non-square counts, and color-mode frequencies, combining each dataset's original train/test splits. These statistics use only successfully decoded images. The complete measurements, including split-specific statistics, unreadable-file counts, and minimum/maximum category sizes, remain available in `image_statistics.json`.

For each dataset, show the 20 most frequent original resolutions, with image counts and percentages of readable images. The complete frequency tables remain available as `dimension_frequencies["plantvillage"]` and `dimension_frequencies["plantdoc"]` in this notebook. PlantVillage source splits and PlantDoc train/test folders are combined within each dataset. Bubble charts plot width against height; marker area increases with the number of images at each resolution (with a minimum size so rare resolutions remain visible). The dashed diagonal represents square images.

Different sizes may require resizing, but a crop policy must avoid losing useful leaf details. Exact SHA-256 duplicates are confirmed **byte-identical**; separate versions of the same leaf need not have identical hashes. PlantVillage leaf IDs are taken from the author's metadata where resolvable. Any possible leakage needs review before splitting data.


In [ ]:
image_statistics = pd.DataFrame(
    json.loads((OUTPUT_DIR / "image_statistics.json").read_text())["partitions"]
)
overall_statistics = image_statistics[image_statistics["split"].eq("all")]

def dimension_range(row, dimension):
    return " / ".join(
        f"{row[f'{dimension}_{stat}']:g}" if pd.notna(row[f"{dimension}_{stat}"]) else "n/a"
        for stat in ("min", "median", "max")
    )

print("Aggregate image dimensions by dataset (min / median / max in pixels):")
dimension_rows = [
    {
        "dataset": DATASET_LABELS.get(row["dataset"], row["dataset"]),
        "images": row["decoded_images"],
        "width (min / median / max)": dimension_range(row, "width"),
        "height (min / median / max)": dimension_range(row, "height"),
        "non-square": row["nonsquare"],
    }
    for row in overall_statistics.to_dict("records")
]
display(pd.DataFrame(dimension_rows, columns=[
    "dataset", "images", "width (min / median / max)",
    "height (min / median / max)", "non-square",
]))

mode_rows = [
    {"dataset": DATASET_LABELS.get(row["dataset"], row["dataset"]),
     "mode": mode, "images": count}
    for row in overall_statistics.to_dict("records")
    for mode, count in row["modes"].items()
]
print("Image color modes by dataset (successfully decoded images):")
display(pd.DataFrame(mode_rows, columns=["dataset", "mode", "images"]))


In [ ]:
dimensions = inventory[inventory["readable"].astype(str).str.lower().eq("true")].copy()
dimensions[["width", "height"]] = dimensions[["width", "height"]].apply(pd.to_numeric)
dimension_frequencies = {}
for dataset, display_name in DATASET_LABELS.items():
    subset = dimensions[dimensions["dataset"] == dataset]
    if subset.empty:
        print(f"{display_name}: no readable images")
        continue

    frequency = (subset.groupby(["width", "height"]).size().rename("images")
                 .reset_index().sort_values(["images", "width", "height"],
                                            ascending=[False, True, True]).reset_index(drop=True))
    frequency["resolution"] = frequency["width"].astype(str) + " × " + frequency["height"].astype(str)
    frequency["percentage"] = (100 * frequency["images"] / len(subset)).round(2)
    dimension_frequencies[dataset] = frequency
    print(f"{display_name}: {len(subset):,} readable images; {len(frequency):,} unique resolutions (top 20 shown)")
    display(frequency[["resolution", "images", "percentage"]].head(20))

    max_count = frequency["images"].max()
    bubble_area = lambda count: 15 + 600 * count / max_count
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(frequency["width"], frequency["height"], s=bubble_area(frequency["images"]),
               alpha=.5, edgecolors="black", linewidths=.3)
    axis_min = min(frequency["width"].min(), frequency["height"].min()) * .8
    axis_max = max(frequency["width"].max(), frequency["height"].max()) * 1.2
    ax.plot([axis_min, axis_max], [axis_min, axis_max], linestyle="--", color="gray",
            linewidth=1, label="Square images")
    if axis_max / axis_min > 10:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.set(xlim=(axis_min, axis_max), ylim=(axis_min, axis_max),
           title=f"{display_name}: image dimensions", xlabel="Width (pixels)", ylabel="Height (pixels)")
    for count in sorted({1, max(1, max_count // 10), max_count}):
        ax.scatter([], [], s=bubble_area(count), alpha=.5, edgecolors="black", linewidths=.3,
                   label=f"{count:,} image(s)")
    ax.legend(title="Bubble-size reference", fontsize=8)
    ax.grid(alpha=.2)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{dataset}_dimension_bubbles.png", dpi=160)
    plt.show()
print("Potential issues:")
if flags.empty:
    print("No automated flags in this run.")
else:
    display(flags.groupby(["dataset", "flag"]).size().rename("flagged_entries").reset_index())
    display(flags)
print("Confirmed byte-identical hash groups:", summary["exact_duplicate_hash_groups"])
print("Cross-partition hash groups:", summary["exact_duplicate_cross_partition_groups"])
print("PlantVillage leaf groups spanning splits:", summary["same_leaf_cross_split_groups"])


### 3a. Exact duplicate groups: impact and supporting flags

Each SHA-256 group with at least two files gets **exactly one** `experiment_impact` value:

| Experiment impact | Meaning | Recommended next step |
| --- | --- | --- |
| `training_test_overlap` | Copies appear in PlantVillage train and at least one planned test set. | Exclude image from training; include image in testing. |
| `duplicate_test` | More than one copy is in the planned test images, without training overlap. | Deduplicate test images. |
| `duplicate_training` | More than one copy is in PlantVillage train, without test overlap. | Deduplicate training images. |
| `no_direct_impact` | The copies do not create duplicate training/test observations under the current plan. | No duplicate-related change needed under the current data-use plan. |

These classifications take precedence **in the order shown**. PlantVillage train is the planned training set; PlantVillage test and PlantDoc test are test sets; PlantDoc train is unused. The final PlantVillage train/validation subdivision does not yet exist.

Three separate Boolean flags describe group properties without changing its primary classification: `label_conflict` (different original categories within a dataset), `cross_dataset` (PlantVillage and PlantDoc both contain the bytes), and `cross_source_split` (train/test overlap within one original dataset). Different names across datasets are **not automatically label conflicts**.

`duplicate_groups.csv` provides a group-level audit and one `recommended_action` per group, with label-review instructions added for conflicts except in the `no_direct_impact` case. `duplicate_group_members.csv` lists all original image paths and whether each is used for `training`, `evaluation`, or is `unused`. These are findings about **exact copies only**; neither source images nor partitions are changed.


In [ ]:
duplicate_groups = pd.read_csv(OUTPUT_DIR / "duplicate_groups.csv", keep_default_na=False)
duplicate_members = pd.read_csv(OUTPUT_DIR / "duplicate_group_members.csv", keep_default_na=False)
impact_order = ["training_test_overlap", "duplicate_test", "duplicate_training", "no_direct_impact"]
print("Exact duplicate groups:", summary["exact_duplicate_hash_groups"])
print("Additional properties:")
print("  Label conflicts:", summary["exact_duplicate_label_conflict_groups"])
print("  Cross-dataset groups:", summary["exact_duplicate_cross_dataset_groups"])
print("  Cross-source-split groups:", summary["exact_duplicate_cross_source_split_groups"])
if duplicate_groups.empty:
    print("No exact duplicate groups found.")
else:
    impact_counts = (duplicate_groups["experiment_impact"].value_counts()
                     .reindex(impact_order, fill_value=0).rename_axis("experiment_impact")
                     .reset_index(name="groups"))
    display(impact_counts)
    overview = duplicate_groups.assign(hash_id=duplicate_groups["sha256"].str[:12])
    display(overview[["hash_id", "copy_count", "source_partitions", "experiment_impact",
                      "training_copies", "evaluation_copies", "unused_copies",
                      "label_conflict", "cross_dataset", "cross_source_split", "recommended_action"]])
    urgent = duplicate_groups[
        duplicate_groups["experiment_impact"].eq("training_test_overlap")
        | duplicate_groups["label_conflict"].astype(str).str.lower().eq("true")
    ]
    if not urgent.empty:
        print("Member paths for training/test overlap or label-conflict groups:")
        display(duplicate_members[duplicate_members["sha256"].isin(urgent["sha256"])]
                .sort_values(["sha256", "dataset", "split", "category", "relative_path"]))


### 3b. Case-insensitive filename collision detection

`analyze_data.py` identifies groups of relative paths that become identical after `casefold()` but differ in capitalization. It writes the affected paths to `case_insensitive_filename_collisions.csv` and flags each member in `quality_flags.csv`. These are filename conflicts on case-insensitive filesystems, **not evidence that image content matches**. Images are not displayed or compared here.


In [ ]:
collision_groups = pd.read_csv(OUTPUT_DIR / "case_insensitive_filename_collisions.csv", keep_default_na=False)
print(f"Case-insensitive filename collision groups: {len(collision_groups):,}")
display(collision_groups)


## 4. Findings to record after the local run

Fill in the following **only after executing and reviewing** the outputs. Keep measured totals separate from published dataset counts.

| Observation (measured) | Evidence (table/figure/filenames) | Proposed preparation step |
| --- | --- | --- |
| Unreadable or unsupported images | To be checked | Flag/exclude unreadable files, retaining their paths |
| Dimensions and potential crop loss | To be checked | Consistent resizing/normalization; choose a crop policy that retains useful leaf detail |
| Identical or related images crossing splits | To be checked | Group duplicates and same-leaf photos before final train/validation/test assignment |
| Class imbalance | To be checked | Report per-category counts and use macro-averaged evaluation later |

**Limits:** Current source train/test names are not the project's final experiment partitions. PlantDoc test must not be used to tune preprocessing or model settings. Original category names are not automatically equivalent across datasets. File and hash checks alone do not validate disease labels. Export selected charts to `docs/figures/` for Troy's report if needed.
